# Анализ личных расходов

Ноутбук читает данные напрямую из `data/expenses.sqlite3`, проверяет качество данных и показывает структуру расходов, динамику по месяцам, категории, дни недели и крупнейшие операции.

**Период и все итоговые значения вычисляются автоматически при запуске**, поэтому отчёт можно обновлять после добавления новых трат в базу.

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (11, 5.5),
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'font.size': 10,
})

def rubles(x, pos=None):
    return f'{x:,.0f} ₽'.replace(',', ' ')

money_fmt = FuncFormatter(rubles)
WEEKDAYS_RU = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

## Загрузка данных

Путь определяется и при запуске из корня проекта, и при запуске из папки `notebooks`. Соединение с SQLite открывается только для чтения.

In [ ]:
candidates = [Path('data/expenses.sqlite3'), Path('../data/expenses.sqlite3')]
db_path = next((p.resolve() for p in candidates if p.exists()), None)
if db_path is None:
    raise FileNotFoundError('Не найден data/expenses.sqlite3')

with sqlite3.connect(f'file:{db_path.as_posix()}?mode=ro', uri=True) as conn:
    expenses = pd.read_sql_query('SELECT * FROM expenses', conn)

expenses['expense_date'] = pd.to_datetime(expenses['expense_date'], errors='coerce')
expenses['created_at'] = pd.to_datetime(expenses['created_at'], errors='coerce', utc=True)
expenses['expense_amount'] = pd.to_numeric(expenses['expense_amount'], errors='coerce')
expenses['expense_description'] = expenses['expense_description'].astype('string').str.lower()
expenses = expenses.sort_values(['expense_date', 'id']).reset_index(drop=True)

print(f'Загружено строк: {len(expenses):,}'.replace(',', ' '))
print(f'Источник: {db_path}')
display(expenses.head())

## Проверка качества данных

In [ ]:
duplicate_business = expenses.duplicated(
    subset=['expense_date', 'expense_amount', 'expense_type', 'expense_description'],
    keep=False,
)

quality = pd.DataFrame({
    'Проверка': [
        'Строки', 'Пропуски в дате', 'Некорректные/неположительные суммы',
        'Пустые категории', 'Повторяющиеся id',
        'Строки в потенциальных бизнес-дубликатах'
    ],
    'Значение': [
        len(expenses), expenses['expense_date'].isna().sum(),
        (expenses['expense_amount'].isna() | (expenses['expense_amount'] <= 0)).sum(),
        expenses['expense_type'].isna().sum() + expenses['expense_type'].astype(str).str.strip().eq('').sum(),
        expenses['id'].duplicated().sum(), duplicate_business.sum(),
    ]
})
display(quality)

if duplicate_business.any():
    display(Markdown('**Примечание:** совпадение даты, суммы, категории и описания не обязательно означает ошибку. Эти строки не удаляются автоматически.'))

## Ключевые показатели

In [ ]:
valid = expenses.dropna(subset=['expense_date', 'expense_amount']).copy()
valid = valid[valid['expense_amount'] > 0]
valid['month'] = valid['expense_date'].dt.to_period('M').dt.to_timestamp()
valid['weekday_num'] = valid['expense_date'].dt.weekday
valid['weekday'] = pd.Categorical(
    valid['weekday_num'].map(dict(enumerate(WEEKDAYS_RU))),
    categories=WEEKDAYS_RU,
    ordered=True,
)

total = valid['expense_amount'].sum()
days_span = (valid['expense_date'].max() - valid['expense_date'].min()).days + 1
active_days = valid['expense_date'].nunique()
monthly = valid.groupby('month', as_index=False)['expense_amount'].sum()
category = (valid.groupby('expense_type', as_index=False)
            .agg(Сумма=('expense_amount', 'sum'), Операций=('id', 'count'))
            .sort_values('Сумма', ascending=False))

kpis = pd.DataFrame({
    'Показатель': ['Период', 'Всего расходов', 'Число операций', 'Средний чек',
                   'Медианный чек', 'Активных дней', 'Среднее за календарный день',
                   'Среднее за полный месяц'],
    'Значение': [
        f"{valid['expense_date'].min():%d.%m.%Y} — {valid['expense_date'].max():%d.%m.%Y}",
        rubles(total), f'{len(valid):,}'.replace(',', ' '), rubles(valid['expense_amount'].mean()),
        rubles(valid['expense_amount'].median()), f'{active_days:,}'.replace(',', ' '),
        rubles(total / days_span),
        rubles(monthly.loc[monthly['month'].dt.days_in_month.eq(valid.groupby('month')['expense_date'].nunique().reindex(monthly['month']).to_numpy()), 'expense_amount'].mean())
        if len(monthly) else '—',
    ]
})
display(kpis)

## Динамика расходов по месяцам

In [ ]:
all_months = pd.date_range(valid['month'].min(), valid['month'].max(), freq='MS')
monthly_full = (monthly.set_index('month').reindex(all_months, fill_value=0)
                .rename_axis('month').reset_index())
monthly_full['rolling_3m'] = monthly_full['expense_amount'].rolling(3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(monthly_full['month'], monthly_full['expense_amount'], width=22, color='#4C78A8', label='Расходы')
ax.plot(monthly_full['month'], monthly_full['rolling_3m'], color='#E45756', marker='o', linewidth=2, label='Среднее за 3 месяца')
ax.set_title('Расходы по месяцам')
ax.set_xlabel('')
ax.set_ylabel('Сумма')
ax.yaxis.set_major_formatter(money_fmt)
ax.legend(frameon=False)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

display(monthly_full.assign(
    Месяц=monthly_full['month'].dt.strftime('%Y-%m'),
    Сумма=monthly_full['expense_amount'].map(rubles),
)[['Месяц', 'Сумма']])

## Категории расходов

In [ ]:
category['Доля'] = category['Сумма'] / category['Сумма'].sum()
category['Накопленная доля'] = category['Доля'].cumsum()

plot_cat = category.sort_values('Сумма')
fig, ax = plt.subplots(figsize=(11, max(5.5, len(plot_cat) * 0.36)))
bars = ax.barh(plot_cat['expense_type'], plot_cat['Сумма'], color='#59A14F')
ax.set_title('Расходы по категориям')
ax.set_xlabel('Сумма')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(money_fmt)
for bar, value in zip(bars, plot_cat['Сумма']):
    ax.text(value, bar.get_y() + bar.get_height()/2, '  ' + rubles(value), va='center', fontsize=9)
ax.margins(x=0.18)
plt.tight_layout()
plt.show()

category_view = category.copy()
category_view['Сумма'] = category_view['Сумма'].map(rubles)
category_view['Доля'] = category_view['Доля'].map(lambda x: f'{x:.1%}')
category_view['Средний чек'] = (category['Сумма'].map(lambda x: x) / category['Операций']).map(rubles)
display(category_view[['expense_type', 'Сумма', 'Доля', 'Операций', 'Средний чек']]
        .rename(columns={'expense_type': 'Категория'}))

## Продукты и питание за последние 12 месяцев

Помесячное сравнение категорий «Продукты базовые», «Вкусности» и «Общепит/Доставка». Показаны последние 12 календарных месяцев, включая последний доступный месяц; он может быть неполным.

In [ ]:
food_categories = ['Продукты базовые', 'Вкусности', 'Общепит/Доставка']
last_month = valid['month'].max()
food_months = pd.date_range(last_month - pd.DateOffset(months=11), last_month, freq='MS')

food_monthly = (
    valid.loc[valid['expense_type'].isin(food_categories)]
    .groupby(['month', 'expense_type'])['expense_amount']
    .sum()
    .unstack(fill_value=0)
    .reindex(index=food_months, columns=food_categories, fill_value=0)
)

colors = ['#4C78A8', '#E45756', '#F2CF5B']
food_plot = food_monthly.copy()
food_plot.index = food_plot.index.strftime('%m.%Y')

ax = food_plot.plot(
    kind='bar',
    figsize=(13, 6),
    width=0.82,
    color=colors,
)
food_total = food_monthly.sum(axis=1)
ax.plot(
    range(len(food_plot)),
    food_total.values,
    color='#222222',
    marker='o',
    linewidth=2.5,
    label='Итого по трём категориям',
    zorder=5,
)
ax.set_title('Расходы на продукты и питание по месяцам')
ax.set_xlabel('')
ax.set_ylabel('Сумма')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
ax.yaxis.set_major_formatter(money_fmt)
ax.legend(title='Категория', frameon=False, ncol=3, loc='upper left')
plt.tight_layout()
plt.show()

food_table = food_monthly.copy()
food_table.index = food_table.index.strftime('%m.%Y')
food_table.index.name = 'Месяц'
display(food_table.map(rubles))

food_totals = food_monthly.sum().sort_values(ascending=False)
display(pd.DataFrame({
    'Категория': food_totals.index,
    'За 12 месяцев': food_totals.map(rubles).values,
    'Среднее в месяц': (food_totals / 12).map(rubles).values,
}))

## Когда происходят траты

In [ ]:
weekday = (valid.groupby('weekday', observed=False)
           .agg(Сумма=('expense_amount', 'sum'), Операций=('id', 'count'))
           .reindex(WEEKDAYS_RU))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].bar(weekday.index, weekday['Сумма'], color='#F28E2B')
axes[0].set_title('Сумма по дням недели')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[1].bar(weekday.index, weekday['Операций'], color='#76B7B2')
axes[1].set_title('Число операций по дням недели')
axes[1].set_ylabel('Операции')
plt.tight_layout()
plt.show()

## Распределение сумм и крупные покупки

In [ ]:
p99 = valid['expense_amount'].quantile(0.99)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].hist(valid['expense_amount'].clip(upper=p99), bins=35, color='#4C78A8', edgecolor='white')
axes[0].set_title('Распределение сумм (до 99-го перцентиля)')
axes[0].set_xlabel('Сумма операции')
axes[0].xaxis.set_major_formatter(money_fmt)
axes[1].boxplot(valid['expense_amount'], vert=True, showfliers=True)
axes[1].set_title('Полное распределение сумм')
axes[1].set_xticks([])
axes[1].yaxis.set_major_formatter(money_fmt)
plt.tight_layout()
plt.show()

top_expenses = valid.nlargest(15, 'expense_amount')[
    ['expense_date', 'expense_amount', 'expense_type', 'expense_description']
].copy()
top_expenses['expense_date'] = top_expenses['expense_date'].dt.strftime('%d.%m.%Y')
top_expenses['expense_amount'] = top_expenses['expense_amount'].map(rubles)
display(top_expenses.rename(columns={
    'expense_date': 'Дата', 'expense_amount': 'Сумма',
    'expense_type': 'Категория', 'expense_description': 'Описание'
}))

## Частые описания

In [ ]:
descriptions = (valid.groupby('expense_description', as_index=False)
                .agg(Операций=('id', 'count'), Сумма=('expense_amount', 'sum'),
                     Средний_чек=('expense_amount', 'mean'))
                .sort_values(['Операций', 'Сумма'], ascending=False)
                .head(20))
descriptions_view = descriptions.copy()
descriptions_view['Сумма'] = descriptions_view['Сумма'].map(rubles)
descriptions_view['Средний_чек'] = descriptions_view['Средний_чек'].map(rubles)
display(descriptions_view.rename(columns={
    'expense_description': 'Описание', 'Средний_чек': 'Средний чек'
}))

## Автоматические выводы

In [ ]:
top_cat = category.iloc[0]
peak_month = monthly_full.loc[monthly_full['expense_amount'].idxmax()]
largest = valid.loc[valid['expense_amount'].idxmax()]
top3_share = category.head(3)['Сумма'].sum() / total
weekend_share = valid.loc[valid['weekday_num'].isin([5, 6]), 'expense_amount'].sum() / total

insights = f'''
- За период **{valid['expense_date'].min():%d.%m.%Y} — {valid['expense_date'].max():%d.%m.%Y}** учтено **{rubles(total)}** в {len(valid):,} операциях.
- Главная категория — **{top_cat['expense_type']}**: {rubles(top_cat['Сумма'])}, или {top_cat['Доля']:.1%} всех расходов.
- Три крупнейшие категории дают **{top3_share:.1%}** общей суммы.
- Максимум за месяц — **{peak_month['month']:%B %Y}**: {rubles(peak_month['expense_amount'])}.
- Самая крупная операция — **{rubles(largest['expense_amount'])}**, {largest['expense_date']:%d.%m.%Y}, категория «{largest['expense_type']}».
- На субботу и воскресенье приходится **{weekend_share:.1%}** общей суммы.
- Средний чек — **{rubles(valid['expense_amount'].mean())}**, медианный — **{rubles(valid['expense_amount'].median())}**. Разница между ними показывает влияние крупных разовых покупок.
'''
display(Markdown(insights))

## Как обновить отчёт

1. Обновите `data/expenses.sqlite3`.
2. Перезапустите все ячейки (`Run All`).
3. Потенциальные дубликаты проверяйте вручную: одинаковые дата, сумма, категория и описание могут быть как дублем, так и двумя реальными покупками.

In [ ]:
expenses.loc[expenses["expense_type"] == "Обед"]["expense_amount"].sum()

In [ ]:
expenses.loc[expenses["expense_date"] > pd.to_datetime("2026-08-30")]